## [Step 3-1] 자연어 텍스트 임베딩 추출 (RoBERTa)

**1. 작업 목표**
머신러닝 모델(CatBoost)은 영어 문장을 그대로 읽을 수 없습니다. 따라서 사전 학습된 거대 언어 모델(LLM)을 활용하여, 리뷰 텍스트 문장의 전반적인 맥락과 감정을 컴퓨터가 이해할 수 있는 **768차원의 숫자 벡터(Vector)** 로 번역하는 작업을 수행합니다.

**2. 세부 설정**
* **사용 모델:** `distilroberta-base` (RoBERTa 모델의 경량화 버전으로, 성능은 유지하면서 연산 속도가 훨씬 빠릅니다.)
* **하드웨어:** T4 GPU (연산 속도 최적화)
* **배치 처리:** GPU 메모리 초과(OOM) 에러를 방지하기 위해 15,000개의 데이터를 32개씩(Batch) 쪼개어 순차적으로 임베딩을 추출합니다.

**3. 예상 결과물**
* `15,000(리뷰 개수) x 768(문맥 벡터)` 크기의 거대한 숫자 행렬 도출

## [Step 3-1] RoBERTa 임베딩 추출: 필라델피아 (Philadelphia)
* **목표:** 필라델피아 리뷰 15,000개의 문맥을 768차원 벡터로 변환
* **입력 데이터:** `yelp_subset_philly_15k.csv`
* **출력 데이터:** `hong_philly_embeddings.npy` (구글 드라이브 자동 저장)

In [ ]:
!pip install transformers torch tqdm

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

# 1. 필라델피아 데이터 로드
file_path = '/content/yelp_subset_philly_15k.csv'
df = pd.read_csv(file_path)
texts = df['text'].astype(str).tolist()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. 모델 로드
model_name = 'distilroberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

# 3. 임베딩 추출 (Batch)
batch_size = 32
embeddings_list = []

with torch.no_grad():
    for i in tqdm(range(0, len(texts), batch_size), desc="[Philly] 임베딩 진행률"):
        batch_texts = texts[i : i + batch_size]
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
        outputs = model(**inputs)
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings_list.append(cls_embeddings)

# 4. 행렬 병합 및 드라이브 안전 저장
philly_embeddings = np.vstack(embeddings_list)
save_path = '/content/drive/MyDrive/ml_project/Team-6/hong_philly_embeddings.npy'
np.save(save_path, philly_embeddings)

print(f"필라델피아 임베딩 완료 및 드라이브 저장 성공! (크기: {philly_embeddings.shape})")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[Philly] 임베딩 진행률: 100%|██████████| 469/469 [03:31<00:00,  2.22it/s]


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/ml_project/Team-6/hong_philly_embeddings.npy'

In [ ]:
import shutil
from google.colab import drive

# 1. 혹시 풀렸을지 모를 구글 드라이브 다시 연결
drive.mount('/content/drive')

# 2. 코랩 내부 저장소(안전지대)에 먼저 파일 저장
local_path = '/content/philly_embeddings.npy'
np.save(local_path, philly_embeddings)
print("1. 코랩 내부 저장 성공!")

# 3. 팀 공유 구글 드라이브 폴더로 파일 복사
drive_path = '/content/drive/MyDrive/ml_project/Team-6/text_embedding/philly_embeddings.npy'
shutil.copy(local_path, drive_path)
print("2. 구글 드라이브 공유 문서함으로 안전하게 복사 완료!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1. 코랩 내부 저장 성공!
2. 구글 드라이브 공유 문서함으로 안전하게 복사 완료!


## [Step 3-2] PCA 차원 축소: 필라델피아 (Philadelphia)
* **목표:** CatBoost 모델의 학습 속도와 효율을 높이기 위해, 필라델피아 데이터의 768차원 문맥 벡터를 핵심 정보만 남기고 32차원으로 압축합니다.
* **출력 데이터:** `philly_pca_32.csv` (`text_embedding` 폴더에 저장)

In [ ]:
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np
import shutil

print("[Philly] PCA 차원 축소를 시작합니다...")

# 1. 32차원으로 압축하는 PCA 모델 세팅
pca = PCA(n_components=32, random_state=42)

# 2. 방금 구출해서 메모리에 살아있는 philly_embeddings 데이터를 바로 압축!
embeddings_pca = pca.fit_transform(philly_embeddings)

# 3. 데이터 손실률 및 크기 확인
explained_variance = np.sum(pca.explained_variance_ratio_) * 100
print(f"압축 완료! 원본 문맥 정보의 약 {explained_variance:.2f}%를 성공적으로 보존했습니다.")
print(f"압축된 데이터 크기: {embeddings_pca.shape} (15000개의 리뷰가 각각 32개의 숫자로 변환됨)")

# 4. 모델링 팀이 쓰기 좋게 데이터프레임으로 변환
pca_columns = [f'roberta_pca_{i}' for i in range(1, 33)]
df_pca = pd.DataFrame(embeddings_pca, columns=pca_columns)

# 5. 코랩 내부에 먼저 CSV로 저장 후 구글 드라이브(text_embedding 폴더)로 복사
local_csv_path = '/content/philly_pca_32.csv'
drive_csv_path = '/content/drive/MyDrive/ml_project/Team-6/text_embedding/philly_pca_32.csv'

df_pca.to_csv(local_csv_path, index=False)
shutil.copy(local_csv_path, drive_csv_path)

print(f"필라델피아 3단계 최종 결과물 CSV 저장 완료: {drive_csv_path}")

[Philly] PCA 차원 축소를 시작합니다...
압축 완료! 원본 문맥 정보의 약 81.44%를 성공적으로 보존했습니다.
압축된 데이터 크기: (15000, 32) (15000개의 리뷰가 각각 32개의 숫자로 변환됨)
필라델피아 3단계 최종 결과물 CSV 저장 완료: /content/drive/MyDrive/ml_project/Team-6/text_embedding/philly_pca_32.csv


## [Step 3-1] RoBERTa 임베딩 추출: 투손 (Tucson)
* **목표:** 투손 리뷰 15,000개의 문맥을 768차원 벡터로 변환
* **출력 데이터:** `tucson_embeddings.npy` (`text_embedding` 폴더에 저장)

In [ ]:
import pandas as pd
import numpy as np
import torch
import shutil
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from google.colab import drive

# 1. 드라이브 마운트 및 연산 장치 확인
drive.mount('/content/drive')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"현재 사용 중인 연산 장치: {device}")

# 2. 데이터 및 모델 로드
file_path = '/content/yelp_subset_tucson_15k.csv'
df = pd.read_csv(file_path)
texts = df['text'].astype(str).tolist()

model_name = 'distilroberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

# 3. 임베딩 추출 (Batch)
batch_size = 32
embeddings_list = []

print("[Tucson] RoBERTa 텍스트 임베딩 추출 시작...")
with torch.no_grad():
    for i in tqdm(range(0, len(texts), batch_size), desc="[Tucson] 임베딩 진행률"):
        batch_texts = texts[i : i + batch_size]
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
        outputs = model(**inputs)
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings_list.append(cls_embeddings)

tucson_embeddings = np.vstack(embeddings_list)

# 4. 코랩 내부 저장 후 드라이브 복사 (연결 끊김 에러 완벽 방지)
local_npy_path = '/content/tucson_embeddings.npy'
drive_npy_path = '/content/drive/MyDrive/ml_project/Team-6/text_embedding/tucson_embeddings.npy'

np.save(local_npy_path, tucson_embeddings)
shutil.copy(local_npy_path, drive_npy_path)

print(f"투손 임베딩 완료 및 안전 저장 성공! (크기: {tucson_embeddings.shape})")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
현재 사용 중인 연산 장치: cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[Tucson] RoBERTa 텍스트 임베딩 추출 시작...


[Tucson] 임베딩 진행률: 100%|██████████| 469/469 [03:26<00:00,  2.27it/s]

투손 임베딩 완료 및 안전 저장 성공! (크기: (15000, 768))


## [Step 3-2] PCA 차원 축소: 투손 (Tucson)
* **목표:** CatBoost 모델의 학습을 위해 투손 데이터의 768차원 벡터를 32차원으로 압축
* **출력 데이터:** `tucson_pca_32.csv` (`text_embedding` 폴더에 저장)

In [ ]:
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np
import shutil

print("[Tucson] PCA 차원 축소를 시작합니다...")

# 1. 32차원으로 압축하는 PCA 세팅
pca = PCA(n_components=32, random_state=42)

# 2. 방금 생성된 tucson_embeddings 데이터를 바로 압축
embeddings_pca = pca.fit_transform(tucson_embeddings)

# 3. 데이터 손실률 확인
explained_variance = np.sum(pca.explained_variance_ratio_) * 100
print(f"압축 완료! 원본 문맥 정보 보존율: {explained_variance:.2f}%")
print(f"압축된 데이터 크기: {embeddings_pca.shape}")

# 4. 데이터프레임으로 변환
pca_columns = [f'roberta_pca_{i}' for i in range(1, 33)]
df_pca = pd.DataFrame(embeddings_pca, columns=pca_columns)

# 5. 안전 저장 (코랩 ➔ 구글 드라이브)
local_csv_path = '/content/tucson_pca_32.csv'
drive_csv_path = '/content/drive/MyDrive/ml_project/Team-6/text_embedding/tucson_pca_32.csv'

df_pca.to_csv(local_csv_path, index=False)
shutil.copy(local_csv_path, drive_csv_path)

print(f"투손 3단계 최종 결과물 CSV 저장 완료: {drive_csv_path}")

[Tucson] PCA 차원 축소를 시작합니다...
압축 완료! 원본 문맥 정보 보존율: 81.45%
압축된 데이터 크기: (15000, 32)
투손 3단계 최종 결과물 CSV 저장 완료: /content/drive/MyDrive/ml_project/Team-6/text_embedding/tucson_pca_32.csv


## [Step 3-1] RoBERTa 임베딩 추출: 뉴올리언스 (New Orleans)
* **목표:** 뉴올리언스 리뷰 15,000개의 문맥을 768차원 벡터로 변환
* **출력 데이터:** `new_orleans_embeddings.npy` (`text_embedding` 폴더에 저장)

In [ ]:
import pandas as pd
import numpy as np
import torch
import shutil
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from google.colab import drive

# 1. 드라이브 마운트 및 연산 장치 확인
drive.mount('/content/drive')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"현재 사용 중인 연산 장치: {device}")

# 2. 데이터 및 모델 로드
file_path = '/content/yelp_subset_new_orleans_15k.csv'
df = pd.read_csv(file_path)
texts = df['text'].astype(str).tolist()

model_name = 'distilroberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

# 3. 임베딩 추출 (Batch)
batch_size = 32
embeddings_list = []

print("[New Orleans] RoBERTa 텍스트 임베딩 추출 시작...")
with torch.no_grad():
    for i in tqdm(range(0, len(texts), batch_size), desc="[New Orleans] 임베딩 진행률"):
        batch_texts = texts[i : i + batch_size]
        inputs = tokenizer(batch_texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
        outputs = model(**inputs)
        cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings_list.append(cls_embeddings)

new_orleans_embeddings = np.vstack(embeddings_list)

# 4. 코랩 내부 저장 후 드라이브 복사 (안전 장치)
local_npy_path = '/content/new_orleans_embeddings.npy'
drive_npy_path = '/content/drive/MyDrive/ml_project/Team-6/text_embedding/new_orleans_embeddings.npy'

np.save(local_npy_path, new_orleans_embeddings)
shutil.copy(local_npy_path, drive_npy_path)

print(f"뉴올리언스 임베딩 완료 및 안전 저장 성공! (크기: {new_orleans_embeddings.shape})")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
현재 사용 중인 연산 장치: cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[New Orleans] RoBERTa 텍스트 임베딩 추출 시작...


[New Orleans] 임베딩 진행률: 100%|██████████| 469/469 [03:26<00:00,  2.27it/s]


뉴올리언스 임베딩 완료 및 안전 저장 성공! (크기: (15000, 768))


## [Step 3-2] PCA 차원 축소: 뉴올리언스 (New Orleans)
* **목표:** CatBoost 모델의 학습 속도 향상을 위해 뉴올리언스 데이터의 768차원 벡터를 32차원으로 압축
* **출력 데이터:** `new_orleans_pca_32.csv` (`text_embedding` 폴더에 저장)

In [ ]:
from sklearn.decomposition import PCA
import pandas as pd
import numpy as np
import shutil

print("[New Orleans] PCA 차원 축소를 시작합니다...")

# 1. 32차원으로 압축하는 PCA 세팅
pca = PCA(n_components=32, random_state=42)

# 2. 방금 생성된 new_orleans_embeddings 데이터를 바로 압축
embeddings_pca = pca.fit_transform(new_orleans_embeddings)

# 3. 데이터 손실률 확인
explained_variance = np.sum(pca.explained_variance_ratio_) * 100
print(f"압축 완료! 원본 문맥 정보 보존율: {explained_variance:.2f}%")
print(f"압축된 데이터 크기: {embeddings_pca.shape}")

# 4. 데이터프레임으로 변환
pca_columns = [f'roberta_pca_{i}' for i in range(1, 33)]
df_pca = pd.DataFrame(embeddings_pca, columns=pca_columns)

# 5. 안전 저장 (코랩 ➔ 구글 드라이브)
local_csv_path = '/content/new_orleans_pca_32.csv'
drive_csv_path = '/content/drive/MyDrive/ml_project/Team-6/text_embedding/new_orleans_pca_32.csv'

df_pca.to_csv(local_csv_path, index=False)
shutil.copy(local_csv_path, drive_csv_path)

print(f"뉴올리언스 3단계 최종 결과물 CSV 저장 완료: {drive_csv_path}")

[New Orleans] PCA 차원 축소를 시작합니다...
압축 완료! 원본 문맥 정보 보존율: 81.69%
압축된 데이터 크기: (15000, 32)
뉴올리언스 3단계 최종 결과물 CSV 저장 완료: /content/drive/MyDrive/ml_project/Team-6/text_embedding/new_orleans_pca_32.csv
